In [2]:
import os
import pandas as pd
import numpy as np
import torch
import torchvision
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from deepml.tasks import ImageRegression
from deepml.train import Learner
from deepml.losses import RMSELoss
import torch.nn as nn

# 1) Subclass ImageRegression to handle a 7-element target
class MultiTargetRegression(ImageRegression):
    """
    This overrides transform_target() and transform_output() so that
    a 7-element tensor is converted into a list of seven rounded floats
    (instead of using y.item() which only works for scalars).
    """
    def transform_target(self, y: torch.Tensor):
        # y is now shape (7,), so do something like [ round(y[i].item(),2) for i in range(7) ]
        return [ round(val.item(), 2) for val in y ]

    def transform_output(self, y: torch.Tensor):
        # y is also shape (7,) for the model’s prediction; do the same formatting
        return [ round(val.item(), 2) for val in y ]


# 2) Everything else is essentially the same as before—but use MultiTargetRegression instead of ImageRegression.

# Custom Dataset returning a 7-vector of raw targets
class MultiTargetImageDataset(Dataset):
    """
    Returns an image plus a 7‐vector of raw targets:
      ["gsw","gtw","VPleaf","VPDleaf","H2O_leaf","Fs","Fm'"]
    """
    def __init__(self, df: pd.DataFrame, feature_column: str, label_columns: list, root_dir: str, transforms=None):
        super().__init__()
        self.df = df.reset_index(drop=True)
        self.feature_column = feature_column
        self.label_columns = label_columns  # length=7
        self.root_dir = root_dir
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # 1) Load image
        img_name = self.df.loc[idx, self.feature_column]
        img_path = os.path.join(self.root_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        # 2) Apply transforms
        if self.transforms is not None:
            image = self.transforms(image)

        # 3) Gather the seven targets into a (7,) tensor
        labels_np = self.df.loc[idx, self.label_columns].values.astype(np.float32)
        labels_tensor = torch.from_numpy(labels_np)  # shape = (7,)
        return image, labels_tensor


# Model setup: final FC now outputs 7 instead of 1
def setup_model_multi7():
    model = torchvision.models.resnet50(pretrained=True)
    for param in model.parameters():
        param.requires_grad = False
    model.fc = nn.Linear(in_features=2048, out_features=7)  # ← 7 outputs
    for param in model.fc.parameters():
        param.requires_grad = True
    model.to(device)
    return model

# Compute mean/std of training images (unchanged)
def compute_dataset_stats(image_dir):
    image_files = [os.path.join(image_dir, f) for f in os.listdir(image_dir)
                   if f.lower().endswith(('png','jpg','jpeg'))]
    rgb_accumulator = []
    for path in image_files:
        arr = np.array(Image.open(path)) / 255.0
        flat = arr.reshape(-1, 3)
        non_black = flat[~np.all(flat == [0,0,0], axis=1)]
        if non_black.size > 0:
            rgb_accumulator.append(non_black)
    rgb_all = np.concatenate(rgb_accumulator, axis=0)
    return rgb_all.mean(axis=0).tolist(), rgb_all.std(axis=0).tolist()

class WeightedHuberLoss(nn.Module):
    def __init__(self, weight_vector: torch.Tensor, delta: float = 0.05):
        super().__init__()
        # we store weights of shape (1,7) so it broadcasts over the batch dimension
        self.register_buffer("w", weight_vector.view(1, -1))
        self.huber_none = nn.HuberLoss(reduction="none", delta=delta)

    def forward(self, preds: torch.Tensor, targets: torch.Tensor):
        # preds, targets: each (batch_size, 7)
        loss_elem = self.huber_none(preds, targets)   # shape = (batch_size, 7)
        weighted = loss_elem * self.w                 # broadcast w across batch
        return weighted.mean()                        # average over batch × 7


# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 3) Main loop over CSV pairs
file_pairs = [
    ("/media/edward/HDD/Workspace/Resnet/LEAF/human/training/"
     "train_resnet-leaf-top-and-angled.csv",
     "/media/edward/HDD/Workspace/Resnet/LEAF/human/validation/"
     "validation_resnet-leaf-top-and-angled.csv"),
]

train_dir = "/media/edward/HDD/Workspace/Resnet/LEAF/human/training/images"
val_dir   = "/media/edward/HDD/Workspace/Resnet/LEAF/human/validation/images"

for train_labels_file, val_labels_file in file_pairs:

    # 3a) Read CSVs: now we have seven target columns plus image_file, etc.
    train_df = pd.read_csv(train_labels_file).astype({
        "gsw":      np.float32,
        "gtw":      np.float32,
        "VPleaf":   np.float32,
        "VPDleaf":  np.float32,
        "H2O_leaf": np.float32,
        "Fs":       np.float32,
        "Fm'":      np.float32,
        "image_file": str
    })
    val_df = pd.read_csv(val_labels_file).astype({
        "gsw":      np.float32,
        "gtw":      np.float32,
        "VPleaf":   np.float32,
        "VPDleaf":  np.float32,
        "H2O_leaf": np.float32,
        "Fs":       np.float32,
        "Fm'":      np.float32,
        "image_file": str
    })

    # 3b) Compute RGB mean/std on train images (unchanged)
    mu_rgb, std_rgb = compute_dataset_stats(train_dir)

    # 3c) Transforms (unchanged)
    train_transforms = T.Compose([
        T.RandomHorizontalFlip(p=0.5),
        T.RandomRotation(degrees=(90, 90)),
        T.ColorJitter(brightness=0.1, saturation=0.25, contrast=0.25, hue=0.04),
        T.ToTensor(),
        T.Normalize(mean=mu_rgb, std=std_rgb),
    ])
    val_transforms = T.Compose([
        T.ToTensor(),
        T.Normalize(mean=mu_rgb, std=std_rgb),
    ])

    # 3d) Build datasets that return a 7‐element raw target tensor
    target_columns = ["gsw","gtw","VPleaf","VPDleaf","H2O_leaf","Fs","Fm'"]
    train_dataset = MultiTargetImageDataset(
        train_df,
        feature_column="image_file",
        label_columns=target_columns,
        root_dir=train_dir,
        transforms=train_transforms
    )
    val_dataset = MultiTargetImageDataset(
        val_df,
        feature_column="image_file",
        label_columns=target_columns,
        root_dir=val_dir,
        transforms=val_transforms
    )

    train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_dataset,   batch_size=2, shuffle=False, num_workers=4, pin_memory=True)

    # 4) Build model that outputs 7 values
    model = setup_model_multi7()

    cols = ["gsw","gtw","VPleaf","VPDleaf","H2O_leaf","Fs","Fm'"]
    std_vals = train_df[cols].std().values   # array of length 7
    raw_w = 1.0 / std_vals               # bigger std → smaller raw_w
    weights = raw_w * (7.0 / raw_w.sum())
    weights_tensor = torch.from_numpy(weights.astype(np.float32)).to(device)  # shape=(7,)


    criterion = WeightedHuberLoss(weights_tensor, delta=0.05)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)  # you can bump LR now

    # 6) Use our custom MultiTargetRegression instead of ImageRegression
    regression_task = MultiTargetRegression(model, "resnet50_multi7")

    learner = Learner(regression_task, optimizer, criterion)

    # 7) Metrics remain unchanged (they will average across all 7 dims)
    metrics = [
        ("mse",   nn.MSELoss()),
        ("rmse",  RMSELoss()),
        ("huber", nn.HuberLoss(reduction='mean', delta=0.05))
    ]

    # 8) Now fit() will no longer call y.item() on a 7-vector, because transform_target
    #     has been overridden to produce a list of 7 rounded floats.
    learner.fit(train_loader, val_loader, epochs=20, metrics=metrics)

    # 9) Save the model
    base_name = os.path.basename(train_labels_file).replace("train_", "").replace("resnet-", "human-")
    base_name = os.path.splitext(base_name)[0]
    save_filename = f"{base_name}-multi7-huber-lr1e7-delta5e2.pt"
    save_path = os.path.join("LEAF", save_filename)
    os.makedirs("LEAF", exist_ok=True)
    torch.save(model.state_dict(), save_path)
    print(f"Model (7-target) saved to {save_path}")


/media/edward/HDD/anaconda3/envs/ResNet/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/media/edward/HDD/anaconda3/envs/ResNet/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1/20:


Validation  : 100%|██████████| 18/18 [00:03<00:00,  5.00it/s, loss=0.0273, mse=35400.8916, rmse=185.7543, huber=4.4935]


Training Loss: 0.0323 Validation Loss: 0.0273 [Saving best validation model]
Epoch 2/20:


Validation  : 100%|██████████| 18/18 [00:03<00:00,  4.91it/s, loss=0.017, mse=32531.1047, rmse=177.8872, huber=4.1956] 


...

Training Loss: 0.0093 Validation Loss: 0.0087 
Epoch 20/20:


Validation  : 100%|██████████| 18/18 [00:03<00:00,  5.17it/s, loss=0.0103, mse=3486.9926, rmse=57.3199, huber=1.2586]


Training Loss: 0.0099 Validation Loss: 0.0103 


Training    : 100%|██████████| 52/52 [00:16<00:00,  3.19it/s, loss=0.0099, mse=4962.3564, rmse=67.8905, huber=1.4445]


Model (7-target) saved to LEAF/human-leaf-top-and-angled-multi7-huber-lr1e7-delta5e2.pt
